### Everything Has a __dict_
Most Python objects store attributes in a dictionary.

In [14]:
class User:
    def __init__(self, name):
        self.name = name
u = User("Nishant")

u.__dict__

{'name': 'Nishant'}

In [15]:
# Attribute Lookup Order
u.name

'Nishant'

Python searches in this order:  
1️⃣ instance.__dict__  
2️⃣ class.__dict__  
3️⃣ parent classes (MRO) 

In [16]:
# Classes also have dictionaries.
class A:
    x = 10

print(A.__dict__)

{'__module__': '__main__', '__firstlineno__': 2, 'x': 10, '__static_attributes__': (), '__dict__': <attribute '__dict__' of 'A' objects>, '__weakref__': <attribute '__weakref__' of 'A' objects>, '__doc__': None}


### __slots__ (Memory Optimization)
#### Normal objects store attributes in __dict__.
Each instance has:
object
+
dictionary

# Python `__slots__` – Advantages & Disadvantages

## Advantages

- **Lower Memory Usage**
  - Removes per-instance `__dict__`, saving significant memory when creating many objects.

- **Faster Attribute Access**
  - Attributes are stored in a fixed structure instead of a dictionary.

- **Prevents Dynamic Attributes**
  - Only predefined attributes can be added, reducing accidental attribute creation.

- **Better for Large-Scale Object Creation**
  - Useful in systems creating millions of objects (e.g., data models, AST nodes).

---

## Disadvantages

- **No Dynamic Attributes**
  - Cannot add new attributes unless defined in `__slots__`.

- **No `__dict__` by Default**
  - Breaks code that relies on `obj.__dict__` or `vars()`.

- **Inheritance Complexity**
  - Subclasses must carefully manage `__slots__`.

- **Multiple Inheritance Issues**
  - May cause layout conflicts if multiple parents define `__slots__`.

- **Harder Debugging / Introspection**
  - Tools relying on attribute dictionaries may not work properly.

- **Serialization Complications**
  - Pickling may require custom `__getstate__` / `__setstate__`.

---

## When to Use

Use `__slots__` when:
- You create **many instances**
- Memory optimization is important
- Object attributes are **fixed**

In [17]:
# Using __slots__
class User:
    __slots__ = ['name']

    def __init__(self, name):
        self.name = name
# Now objects do not have __dict__.
# Memory becomes smaller.

u = User("Nishant")
print(u.__dict__)

# Why frameworks use it
# When you have millions of objects, saving memory matters.

AttributeError: 'User' object has no attribute '__dict__'

Descriptors  
A descriptor is any object implementing:  
__get__()
__set__()
__delete__()

# Descriptors in Python

Descriptors are objects that control how attributes are accessed, set, or deleted in another class using special methods.

They implement one or more of these methods:
- `__get__(self, obj, objtype)`
- `__set__(self, obj, value)`
- `__delete__(self, obj)`

---

## Why Descriptors Are Used

- Customize attribute access
- Implement validation
- Create computed attributes
- Build frameworks (ORMs, properties, etc.)

Examples in Python that use descriptors:
- `property`
- `staticmethod`
- `classmethod`

---

## Example

```python
class PositiveNumber:
    def __get__(self, obj, objtype):
        return obj._value

    def __set__(self, obj, value):
        if value < 0:
            raise ValueError("Value must be positive")
        obj._value = value


class Product:
    price = PositiveNumber()


p = Product()
p.price = 10
print(p.price)

In [ ]:
class Descriptor:
    def __get__(self, obj, objtype):
        print("Getting value")

    def __set__(self, obj, value):
        print("Setting value")
        
class A:
    x = Descriptor()

a = A()
a.x

Getting value


In [ ]:
# @property Uses Descriptors
class User:
    def __init__(self):
        self._age = 20

    @property
    def age(self):
        return self._age
u= User()
print(u.age)

# Python actually does:
User.__dict__['age'].__get__(u, User)

20


20

In [ ]:
class IntegerField:
    def __get__(self, obj, objtype):
        return obj.__dict__.get("age")

    def __set__(self, obj, value):
        if not isinstance(value, int):
            raise TypeError("Age must be an integer")
        obj.__dict__["age"] = value


class User:
    age = IntegerField()


u = User()
u.age = 30
print(u.age)


# Why ORMs use descriptors
#     When you write:
#       user.age = 30

# The ORM can:
    # validate type
    # track changes
    # prepare SQL queries
    # mark field as updated

30


## Example 3: Lazy Loading with Descriptor

Load data only when the attribute is accessed (useful for database or API calls).

```python
class LazyProfile:
    def __get__(self, obj, objtype):
        print("Loading profile from database...")
        return {"name": "Nishant", "age": 25}


class User:
    profile = LazyProfile()


u = User()

print(u.profile)

Use Case:  
Lazy loading is useful when:  
Data is expensive to fetch (database/API)  
You want to load it only when needed  
Improves performance and memory usage

# Method Binding in Python

Method binding is how Python automatically attaches a **method to an instance** when it is accessed.

When a method is called from an object, Python **binds the instance (`self`) to the method automatically**.

---

# Example

```python
class Person:
    def greet(self):
        print("Hello")


p = Person()
p.greet()

In [ ]:
# Person.greet(p)
# Internally Python does something like

# greet is a function defined in the class
# Python binds the instance p to the function
# That becomes a bound method

# MRO (Method Resolution Order) in Python

MRO defines **the order in which Python searches for methods and attributes in a class hierarchy**, especially during **multiple inheritance**.

Python uses the **C3 Linearization Algorithm** to compute MRO.

---

# Example

```python
class A:
    def show(self):
        print("A")

class B(A):
    pass

class C(B):
    pass


c = C()
c.show()

In [18]:
class A:
    def show(self):
        print("A")

class B(A):
    pass

class C(A):
    pass

class D(B, C):
    pass


d = D()
d.show()

A


In [ ]:
print(D.mro())

[<class '__main__.D'>, <class '__main__.B'>, <class '__main__.C'>, <class '__main__.A'>, <class 'object'>]


In [ ]:
#     A
#    / \
#   B   C
#    \ /
#     D
# Diamond problem

# MRO ensures A is called only once:
# super().method()
# super() follows the MRO order, not just the parent class.

class A:
    def show(self):
        print("A")

class B(A):
    pass

class C(A):
    pass

class D(B, C):
    pass

# Metaclasses in Python

A **metaclass** is the class that defines how a **class itself is created**.

Simple hierarchy:
object → class → metaclass

In [20]:
# Creating a Custom Metaclass
# A metaclass is defined by inheriting from type.
class MyMeta(type):
    def __new__(cls, name, bases, attrs):
        print("Creating class:", name)
        return super().__new__(cls, name, bases, attrs)


class Test(metaclass=MyMeta):
    x = 10

Creating class: Test


In [21]:
class UpperAttrMeta(type):

    def __new__(cls, name, bases, attrs):
        new_attrs = {}

        for key, value in attrs.items():
            if not key.startswith("__"):
                new_attrs[key.upper()] = value
            else:
                new_attrs[key] = value

        return super().__new__(cls, name, bases, new_attrs)


class MyClass(metaclass=UpperAttrMeta):
    x = 10
    y = 20


print(MyClass.X)
print(MyClass.Y)

10
20


In [ ]:
# Real World Use: ORMs

# Frameworks like Django ORM / SQLAlchemy use metaclasses.

# Example model:

class User(Model):
    name = CharField()
    age = IntegerField()

# Metaclass responsibilities:
# collect field definitions
# build database schema
# map class fields to database columns
# prepare query logic

In [ ]:
# Class Creation Flow

# When Python sees:
# class A:
#     x = 10

# Steps:
#     1. Collect attributes (x)
#     2. Call metaclass
#       metaclass.__new__()
#      metaclass.__init__()
#     3 .Return class object A

# Metaclass = a class that controls how other classes are created and modified.

In [ ]:
# Mental Model

# object
#   │
#   ├── attributes stored in __dict__
#   │
#   ├── class defines behavior
#   │
#   └── descriptors control attribute access